# EfficientNet-B0 — Four-Breed Dog Classification (PyTorch)

From-scratch EfficientNet-B0 on the **same** train/val/test split as the Custom CNN (seed 42).

**Model:** custom `EfficientNetB0` in `src/model.py` (no ImageNet weights) → 4-class head  
**Stages:** (A) train all layers (lr 1e-3) → (B) fine-tune all layers (lr 1e-4, cosine)

**How to run:** select `EfficientNet/.venv` (or `cnn/.venv`), then **Run All**.  
Outputs go to `EfficientNet/outputs/`.

In [ ]:
# 1. Setup
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

print("Python:", sys.executable)

SEED = 42
BATCH_SIZE = 32
LABEL_SMOOTHING = 0.1
STAGE_A_MAX_EPOCHS = 40
STAGE_A_PATIENCE = 8
STAGE_A_LR = 1e-3
STAGE_B_MAX_EPOCHS = 20
STAGE_B_PATIENCE = 5
STAGE_B_LR = 1e-4
CLASS_NAMES = ["French_bulldog", "Cardigan", "Labrador_retriever", "golden_retriever"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_FIXED = Path("/Users/rexfufung/Desktop/AI Group project /EfficientNet").resolve()
EFF_DIR = _FIXED if (_FIXED / "outputs" / "train.csv").is_file() else Path.cwd().resolve()
if EFF_DIR.name != "EfficientNet" and (EFF_DIR / "EfficientNet" / "outputs" / "train.csv").is_file():
    EFF_DIR = EFF_DIR / "EfficientNet"
OUTPUT_DIR = EFF_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

set_seed(SEED)
device = get_device()
print(f"EFF_DIR={EFF_DIR}")
print(f"Device={device}  torch={torch.__version__}")

## 2. Shared split

Reuse CSVs copied from `cnn/outputs/` (seed 42). Do not re-split.

In [ ]:
# 2. Confirm split CSVs
for name in ("train", "val", "test"):
    df = pd.read_csv(OUTPUT_DIR / f"{name}.csv")
    print(f"{name}: {len(df)}  labels={sorted(df['label'].unique().tolist())}")
meta = json.loads((OUTPUT_DIR / "split_meta.json").read_text())
print("seed:", meta.get("seed"))
print("class_counts:", json.dumps(meta.get("class_counts"), indent=2))

## 3. Preprocessing

ImageNet mean/std, 224×224. Train: RandomResizedCrop + flip + light ColorJitter.

In [ ]:
# 3. Dataset + loaders
class DogBreedDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["path"]) as im:
            image = im.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"])

def build_transforms(train: bool):
    normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.0),
            transforms.ToTensor(),
            normalize,
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize,
    ])

train_loader = DataLoader(DogBreedDataset(OUTPUT_DIR/"train.csv", build_transforms(True)), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(DogBreedDataset(OUTPUT_DIR/"val.csv", build_transforms(False)), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(DogBreedDataset(OUTPUT_DIR/"test.csv", build_transforms(False)), batch_size=BATCH_SIZE, shuffle=False)
xb, yb = next(iter(train_loader))
print("batch", tuple(xb.shape), yb[:8].tolist())

## 4. EfficientNet-B0 (from scratch)

Custom `EfficientNetB0` via `build_efficientnet_b0`. No ImageNet weights.

In [ ]:
# 4. Model (from-scratch EfficientNet-B0)
sys.path.insert(0, str(EFF_DIR / "src"))
from model import build_efficientnet_b0, set_backbone_trainable  # noqa: E402

_m = build_efficientnet_b0(num_classes=4, pretrained=False).to(device)
with torch.no_grad():
    out = _m(xb.to(device))
print(f"params={sum(p.numel() for p in _m.parameters()):,}  logits={tuple(out.shape)}")
del _m

## 5. Train then fine-tune (from scratch)

A: train all layers (lr 1e-3, max 40, patience 8, cosine).  
B: fine-tune all layers (lr 1e-4, max 20, patience 5, cosine).

In [ ]:
# 5. Train
@torch.no_grad()
def evaluate(model, loader, device, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        preds = logits.argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return (
        total_loss / max(n, 1),
        accuracy_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average="macro", zero_division=0),
        confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASS_NAMES)))),
    )

def train_epochs(model, train_loader, val_loader, device, optimizer, criterion, max_epochs, patience, scheduler, history, stage_name):
    best_f1, best_epoch, best_state = -1.0, 0, None
    patience_left = patience if patience is not None else 10**9
    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        if scheduler is not None:
            scheduler.step()
        train_loss, train_acc = running_loss / max(total, 1), correct / max(total, 1)
        val_loss, val_acc, val_f1, _ = evaluate(model, val_loader, device, criterion)
        for k, v in [("train_loss", train_loss), ("train_acc", train_acc), ("val_loss", val_loss), ("val_acc", val_acc), ("val_macro_f1", val_f1)]:
            history[k].append(v)
        history["stage"].append(stage_name)
        print(f"  [{stage_name}] {epoch:02d}/{max_epochs}  train_acc={train_acc:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience if patience is not None else 10**9
        else:
            patience_left -= 1
            if patience is not None and patience_left <= 0:
                print(f"  early stop @ {epoch} (best f1={best_f1:.4f})")
                break
    model.load_state_dict(best_state)
    return best_f1, best_epoch, best_state

set_seed(SEED)
model = build_efficientnet_b0(num_classes=4, pretrained=False).to(device)
set_backbone_trainable(model, True)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
history = {k: [] for k in ["train_loss","train_acc","val_loss","val_acc","val_macro_f1","stage"]}

print("=== Stage A: train all from scratch ===")
opt_a = torch.optim.Adam(model.parameters(), lr=STAGE_A_LR)
sched_a = CosineAnnealingLR(opt_a, T_max=STAGE_A_MAX_EPOCHS)
best_a, epoch_a, state_a = train_epochs(model, train_loader, val_loader, device, opt_a, criterion, STAGE_A_MAX_EPOCHS, STAGE_A_PATIENCE, sched_a, history, "A")
model.load_state_dict(state_a)

print("=== Stage B: fine-tune all (lower LR) ===")
opt_b = torch.optim.Adam(model.parameters(), lr=STAGE_B_LR)
sched_b = CosineAnnealingLR(opt_b, T_max=STAGE_B_MAX_EPOCHS)
best_b, epoch_b, best_state = train_epochs(model, train_loader, val_loader, device, opt_b, criterion, STAGE_B_MAX_EPOCHS, STAGE_B_PATIENCE, sched_b, history, "B")
model.load_state_dict(best_state)

ckpt_path = OUTPUT_DIR / "best_efficientnet_b0.pt"
torch.save({
    "state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "seed": SEED,
    "architecture": "efficientnet_b0_from_scratch",
    "pretrained": None,
    "best_val_macro_f1": best_b,
    "best_stage_b_epoch": epoch_b,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
}, ckpt_path)
print("Saved", ckpt_path, "val_f1", best_b)

## 6. Curves + one-shot test

In [ ]:
# 6. Evaluate
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(epochs, history["train_loss"], label="train"); axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(epochs, history["train_acc"], label="train acc")
axes[1].plot(epochs, history["val_acc"], label="val acc")
axes[1].plot(epochs, history["val_macro_f1"], label="val macro-F1")
axes[1].set_title("Accuracy / Macro-F1"); axes[1].legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=150); plt.show()

test_loss, test_acc, test_f1, cm = evaluate(model, test_loader, device, nn.CrossEntropyLoss())
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right"); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Test CM — EfficientNet-B0")
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150); plt.show()

meta = json.loads((OUTPUT_DIR / "split_meta.json").read_text())
metrics = {
    "seed": SEED, "device": str(device), "architecture": "efficientnet_b0_from_scratch", "pretrained": None,
    "stage_a": {"best_val_macro_f1": best_a, "best_epoch": epoch_a, "lr": STAGE_A_LR},
    "stage_b": {"best_val_macro_f1": best_b, "best_epoch": epoch_b, "lr": STAGE_B_LR},
    "test": {"loss": float(test_loss), "accuracy": float(test_acc), "macro_f1": float(test_f1),
             "confusion_matrix": cm.tolist(), "class_names": CLASS_NAMES},
    "class_counts": meta.get("class_counts"),
    "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"TEST accuracy={test_acc:.4f} macro_f1={test_f1:.4f}")
print(cm)